<a href="https://colab.research.google.com/github/hyperpipe-kr/colab-examples/blob/main/06_BERTopic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BERTopic이란?

BERTopic은 BERT 기반 토픽모델링 기법입니다. BERTopic은 transformers와 c-TF-IDF룰 활용하여 주제 설명에 중요한 단어를 유지하면서도 쉽게 해석할 수 있는 주제를 제공합니다.



BERTopic 구조

BERTopic의 구조는 크게 3가지로 단계로 볼 수 있습니다.

1) 각 문서에 대해 embedding합니다.

2) 각 문서의 벡터 차원을 축소하고 클러스터링 합니다.

3) 클러스터별 토픽을 추출합니다.

- Embed Document(문서 임베딩): 사전학습된 Transformer 모델을 사용하여 텍스트 데이터를 수치 벡터로 변환합니다.
- UMAP(차원 축소): 고차원의 임베딩 벡터를 축소하여 클러스터링이 가능하도록 만들어 줍니다.
- HDBSCAN(클러스터링): 의미적으로 유사한 문서를 하나의 클러스터로 묶어주며, 클러스터(토픽) 개수를 미리 지정할 필요 없고 노이즈 처리가 가능합니다.
- c-TF-IDF(class-based TF-IDF): 각 클러스터(토픽)를 대표하는 단어를 추출합니다.
- MMR(토픽 키워드 최적화): 유사성 없이, 의미적으로 다양한 키워드를 선택합니다.

BERTopic 영화 리뷰 코드 실행

In [ ]:
!pip install bertopic sentence-transformers nltk -q

import nltk
nltk.download('movie_reviews')
nltk.download('stopwords')

from nltk.corpus import movie_reviews, stopwords
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
import re

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 3.2 MB/s eta 0:00:00


[nltk_data] Downloading package movie_reviews to /root/nltk_data...
[nltk_data]   Unzipping corpora/movie_reviews.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


1. 데이터 준비

In [ ]:
documents = [movie_reviews.raw(fileid) for fileid in movie_reviews.fileids()]

2. 데이터 전처리

정규 표현식: 소문자화, 3자 이상 단어만 추출
불용어: nltk 불용어 셋 사용하여 제거

In [ ]:
# 전처리 (정규 표현식: 소문자화, 3자 이상 단어 & 불용어: nltk 불용어 셋)
stop_words = set(stopwords.words('english'))

def preprocess(text):
    tokens = re.findall(r'\b[a-zA-Z]{3,}\b', text.lower())
    filtered = [word for word in tokens if word not in stop_words]
    return " ".join(filtered)

processed_docs = [preprocess(doc) for doc in documents]

3. 임베딩 모델 준비

In [ ]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

4. BERTopic 모델 생성 및 학습

In [ ]:
topic_model = BERTopic(embedding_model=embedding_model, min_topic_size=5, verbose=True)
topics, probs = topic_model.fit_transform(processed_docs)

print(topic_model.get_topic_info())

2026-07-23 15:03:36,271 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/63 [00:00<?, ?it/s]

2026-07-23 15:08:20,773 - BERTopic - Embedding - Completed ✓
2026-07-23 15:08:20,776 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-07-23 15:08:43,270 - BERTopic - Dimensionality - Completed ✓
2026-07-23 15:08:43,272 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-07-23 15:08:43,375 - BERTopic - Cluster - Completed ✓
2026-07-23 15:08:43,389 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-07-23 15:08:44,181 - BERTopic - Representation - Completed ✓


     Topic  Count                            Name  \
0       -1    910          -1_film_one_movie_like   
1        0     51       0_cop_carter_stallone_joe   
2        1     38      1_chan_jackie_hong_martial   
3        2     26       2_titanic_ship_whale_rose   
4        3     25         3_band_music_blues_rock   
..     ...    ...                             ...   
96      95      5    95_species_patrick_alien_eve   
97      96      5  96_winner_office_philip_edison   
98      97      5     97_wild_things_sex_richards   
99      98      5  98_enemy_smith_future_humanist   
100     99      5      99_gibson_mel_mullen_jerry   

                                        Representation  \
0    [film, one, movie, like, good, even, time, sto...   
1    [cop, carter, stallone, joe, metro, action, sw...   
2    [chan, jackie, hong, martial, chinese, arts, d...   
3    [titanic, ship, whale, rose, cameron, frankens...   
4    [band, music, blues, rock, elwood, breakdown, ...   
..             

5. 시각화

- 토픽 시각화: 군집 간의 거리와 토픽 빈도를 표시하며, 원의 크기는 빈도, 각 원 간의 거리는 토픽 간 의미적 유사성을 의미합니다.
- 단어 시각화: 토픽별 가장 중요한 단어들을 막대 그래프로 시각화합니다.
- 토픽 유사도 시각화: 토픽들 사이의 유사도를 계층적 클러스터링 형태로 시각화하여 가까운 가지일수록 의미적으로 유사한 토픽을 의미합니다.

In [ ]:
# 토픽 시각화
topic_model.visualize_topics()

In [ ]:
# 단어 시각화
topic_model.visualize_barchart(top_n_topics=5, n_words=10)

In [ ]:
# 토픽 유사도 시각화(히트맵)
topic_model.visualize_heatmap()